# Social epistasis model-comparison figures

This notebook generates the combined ΔAIC figures used to compare alternative models of indirect genetic effects across three phenotypes:

- ear-hole area,
- forced swim test immobility during the first 2 minutes, and
- forced swim test immobility during the last 4 minutes.

Two related model sets are visualized separately: the primary social-epistasis models and the extended set that includes fighting-related models.

The notebook expects the model-comparison tables to have already been generated and saved as Excel files. No statistical models are fitted here; this notebook reads those results and creates publication figures.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# Project-relative paths. If the notebook is launched from the repository root,
# these defaults should work without modification.
PROJECT_ROOT = Path.cwd()
STATISTICS_DIR = PROJECT_ROOT / "results" / "statistics" / "social_epistasis"
FIGURE_DIR = PROJECT_ROOT / "results" / "figures" / "social_epistasis"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Input tables: primary social-epistasis model comparison
WH_AIC_FILE = STATISTICS_DIR / "AIC_table_WH_Sex_Strain_Genotype.xlsx"
FST2_AIC_FILE = STATISTICS_DIR / "AIC_table_FST_2mins_Sex_Strain_Genotype_newmethod.xlsx"
FST4_AIC_FILE = STATISTICS_DIR / "AIC_table_FST_4mins_Sex_Strain_Genotype_newmethod.xlsx"

# Input tables: extended model comparison including fighting-related models
WH_AIC_FIGHTING_FILE = STATISTICS_DIR / "AIC_table_WH_Sex_Strain_Genotype_newmethod_fighting.xlsx"
FST2_AIC_FIGHTING_FILE = STATISTICS_DIR / "AIC_table_FST_2mins_Sex_Strain_Genotype_newmethod_fighting.xlsx"
FST4_AIC_FIGHTING_FILE = STATISTICS_DIR / "AIC_table_FST_4mins_Sex_Strain_Genotype_newmethod_fighting.xlsx"


## 2. Extended model comparison including fighting-related models

Repeat the combined ΔAIC visualization using the extended model set that includes the fighting-related alternatives. This section uses separate input tables but otherwise follows the same plotting logic as the primary comparison above.


In [ ]:
wh_aic_df_f = pd.read_excel(WH_AIC_FIGHTING_FILE)
fst2mins_aic_df_f = pd.read_excel(FST2_AIC_FIGHTING_FILE)
fst4mins_aic_df_f = pd.read_excel(FST4_AIC_FIGHTING_FILE)


In [ ]:
# ============================================================
# Global font
# ============================================================

mpl.rcParams["font.family"] = "Arial"


# ============================================================
# 1. Combine the three phenotype dataframes
# ============================================================

plot_df = pd.concat(
    [
        wh_aic_df_f.assign(Phenotype="Ear hole area"),
        fst2mins_aic_df_f.assign(Phenotype="FST 2 min"),
        fst4mins_aic_df_f.assign(Phenotype="FST 4 min"),
    ],
    ignore_index=True
)

# Make sure ΔAIC is numeric
plot_df["ΔAIC"] = pd.to_numeric(
    plot_df["ΔAIC"],
    errors="raise"
)
# ============================================================
# Clean model IDs
# ============================================================

plot_df["Model"] = (
    plot_df["Model"]
    .astype(str)
    .str.strip()
)

# Extract m0, m1, ..., m11
plot_df["Model_ID"] = (
    plot_df["Model"]
    .str.extract(r"^(m\d+)", expand=False)
    .str.lower()
)


# ============================================================
# Model labels
# ============================================================

label_map = {
    "m0": "Null",
    "m1": "Strain",
    "m2": "Sex",
    "m3": "Strain + Sex",
    "m4": "Strain × Sex",
    "m5": "Strain + Sex + Genotype",
    "m6": "Strain + Sex × Genotype",
    "m8": "Strain × Sex + Genotype",
    "m7": "Strain × Genotype + Sex",

    "m10": "SimilarityDependentEpistasis1",

    
    "m9": "Strain × Genotype × Sex",

    "m11": "SimilarityDependentEpistasis2",
}

plot_df["Model_label"] = plot_df["Model_ID"].map(label_map)


# ============================================================
# Exact desired order
# ============================================================

model_id_order = [
    "m0",
    "m1",
    "m2",
    "m3",
    "m4",
    "m5",
    "m6",
    "m8",
    "m7",
    "m10",
    
    "m9",
    "m11",
]

model_label_order = [
    label_map[m]
    for m in model_id_order
]

plot_df["Model_label"] = pd.Categorical(
    plot_df["Model_label"],
    categories=model_label_order,
    ordered=True
)
# ============================================================
# 5. Quick checks
# ============================================================

print(
    plot_df[
        ["Phenotype", "Model", "Model_ID", "Model_label", "ΔAIC"]
    ]
    .sort_values(["Phenotype", "Model_label"])
    .to_string(index=False)
)

print("\nModels per phenotype:")
print(
    plot_df.groupby(
        "Phenotype",
        observed=True
    )["Model_label"].nunique()
)

print("\nModels actually available:")
print(model_order_available)

print("\nMinimum ΔAIC per phenotype:")
print(
    plot_df.groupby(
        "Phenotype",
        observed=True
    )["ΔAIC"].min()
)

print("\nBest model per phenotype:")
print(
    plot_df.loc[
        plot_df.groupby("Phenotype")["ΔAIC"].idxmin(),
        ["Phenotype", "Model_label", "ΔAIC"]
    ]
)

print("\nMaximum ΔAIC per phenotype:")
print(
    plot_df.groupby(
        "Phenotype",
        observed=True
    )["ΔAIC"].max()
)
# ============================================================
# 6. Plot settings
# ============================================================

phenotypes = [
    "FST 2 min",
    "FST 4 min",
    "Ear hole area"
]

# Same displayed scale for all panels
xmax = 15

# Accent color for models within ΔAIC <= 2
support_color = "#D55E00"

# Neutral color for remaining models
neutral_color = "0.35"


fig, axes = plt.subplots(
    nrows=1,
    ncols=3,
    figsize=(12.5, 8.0),
    sharex=True,
    sharey=True
)


# ============================================================
# 7. Draw each phenotype panel
# ============================================================

for ax, phenotype in zip(axes, phenotypes):

    d = (
        plot_df.loc[
            plot_df["Phenotype"] == phenotype
        ]
        .sort_values("Model_label")
        .reset_index(drop=True)
    )

    y = np.arange(len(d))


    # ------------------------------------------
    # Points within displayed range
    # ------------------------------------------

    inside = d["ΔAIC"] <= xmax

    # Within ΔAIC <= 2
    supported = inside & (d["ΔAIC"] <= 2)

    # ΔAIC > 2 but still within plotted scale
    less_supported = inside & (d["ΔAIC"] > 2)


    # Highlight ΔAIC <= 2
    ax.scatter(
        d.loc[supported, "ΔAIC"],
        y[supported],
        s=75,
        color=support_color,
        zorder=4
    )


    # Remaining visible models
    ax.scatter(
        d.loc[less_supported, "ΔAIC"],
        y[less_supported],
        s=60,
        color=neutral_color,
        zorder=3
    )


    # ------------------------------------------
    # Values beyond displayed range
    # ------------------------------------------

    outside = d["ΔAIC"] > xmax

    ax.scatter(
        np.repeat(xmax, outside.sum()),
        y[outside],
        marker=">",
        s=120,
        linewidths=1.8,
        color=neutral_color,
        zorder=5
    )


    # Add actual ΔAIC beside truncated observations
    for yi, value in zip(
        y[outside],
        d.loc[outside, "ΔAIC"]
    ):
        ax.annotate(
            f"{value:.0f}",
            xy=(xmax, yi),
            xytext=(-8, 9),
            textcoords="offset points",
            ha="right",
            va="bottom",
            fontsize=9,
            fontweight="bold",
            color=neutral_color
        )


    # ------------------------------------------
    # ΔAIC = 2 reference line
    # ------------------------------------------

    ax.axvline(
        x=2,
        color=support_color,
        linestyle=":",
        linewidth=1.8,
        alpha=0.95,
        zorder=2
    )


    # ------------------------------------------
    # Axes
    # ------------------------------------------

    ax.set_xlim(-0.5, xmax + 0.6)

    ax.set_yticks(y)

    ax.set_yticklabels(
        d["Model_label"]
    )

    # First model at top
    ax.invert_yaxis()

    ax.set_title(
        phenotype,
        fontsize=16,
        fontweight="bold"
    )

    ax.set_xlabel(
        "ΔAIC",
        fontsize=14,
        fontweight="bold"
    )

    # Horizontal guides only
    ax.grid(
        axis="y",
        linewidth=0.5,
        alpha=0.2
    )

    # Clean paper style
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.tick_params(
        axis="x",
        labelsize=12
    )

    ax.tick_params(
        axis="y",
        labelsize=11
    )


# ============================================================
# 8. Final formatting
# ============================================================

axes[0].set_ylabel(
    "Candidate models",
    fontsize=14,
    fontweight="bold",
    labelpad=15
)

plt.subplots_adjust(
    left=0.33,
    right=0.98,
    bottom=0.11,
    top=0.90,
    wspace=0.12
)


# ============================================================
# 9. Save
# ============================================================
"""
plt.savefig(
    FIGURE_DIR / "AIC_combined_background.pdf",
    format="pdf",
    bbox_inches="tight"
)

plt.savefig(
    FIGURE_DIR / "AIC_combined_background.svg",
    format="svg",
    bbox_inches="tight"
)

plt.close()
"""
plt.show()
